# Cat vs Dog classification with a CNN

Images live under `training_set/training_set` and `test_set/test_set` (subfolders `cats` / `dogs`). A small **convolutional neural network** in PyTorch learns spatial patterns instead of flattening pixels. The training folder is split 80/20 for train vs validation; the test folder is used only for a final evaluation pass.

In [4]:
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Dataset
from torchvision import datasets, transforms

ROOT = Path(".").resolve()
TRAIN_DIR = ROOT / "training_set" / "training_set"
TEST_DIR = ROOT / "test_set" / "test_set"

IMG_SIZE = 128
BATCH_SIZE = 32
SEED = 42
VAL_FRACTION = 0.2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

train_tf = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)
eval_tf = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


class ApplyTransform(Dataset):
    """Use different transforms on train vs val splits of the same ImageFolder."""

    def __init__(self, subset: Dataset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx: int):
        pil_img, label = self.subset[idx]
        return self.transform(pil_img), label


_base_train = datasets.ImageFolder(str(TRAIN_DIR), transform=None)
n_total = len(_base_train)
n_val = max(1, int(n_total * VAL_FRACTION))
n_train = n_total - n_val
generator = torch.Generator().manual_seed(SEED)
train_sub, val_sub = random_split(
    _base_train, [n_train, n_val], generator=generator
)
train_ds = ApplyTransform(train_sub, train_tf)
val_ds = ApplyTransform(val_sub, eval_tf)
test_ds = datasets.ImageFolder(str(TEST_DIR), transform=eval_tf)

class_names = _base_train.classes
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

print(f"Device: {device}")
print(f"Classes: {class_names}")
print(f"Train / val / test sizes: {len(train_ds)}, {len(val_ds)}, {len(test_ds)}")

Device: cuda
Classes: ['cats', 'dogs']
Train / val / test sizes: 6404, 1601, 2023


In [ ]:
class CatDogCNN(nn.Module):

    def __init__(self, spatial: int = IMG_SIZE):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        k = spatial // (2**3)
        flat = 128 * k * k
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(flat, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.40),
            nn.Linear(128, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.head(x).squeeze(-1)


def accuracy_from_logits(logits: torch.Tensor, y: torch.Tensor) -> float:
    preds = (torch.sigmoid(logits) >= 0.5).long()
    return (preds == y).float().mean().item()


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    n_batches = 0
    correct = 0
    total = 0
    criterion = nn.BCEWithLogitsLoss()
    for X, y in loader:
        X = X.to(device)
        y = y.to(device).float()
        logits = model(X)
        total_loss += criterion(logits, y).item()
        n_batches += 1
        preds = (torch.sigmoid(logits) >= 0.5).long()
        correct += (preds == y.long()).sum().item()
        total += y.size(0)
    return total_loss / max(n_batches, 1), correct / max(total, 1)


model = CatDogCNN().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()
EPOCHS = 15
best_val = float("inf")
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    n_batches = 0
    train_acc_batches = []

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device).float()

        opt.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        opt.step()

        running += loss.item()
        n_batches += 1
        train_acc_batches.append(accuracy_from_logits(logits.detach(), y.long()))

    val_loss, val_acc = evaluate(model, val_loader)
    train_loss_epoch = running / max(n_batches, 1)
    avg_train_acc = sum(train_acc_batches) / max(len(train_acc_batches), 1)

    print(
        f"Epoch {epoch:02d}/{EPOCHS}  "
        f"train_loss={train_loss_epoch:.4f}  train_acc_batch_mean={avg_train_acc:.4f}  "
        f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}"
    )

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

test_loss, test_acc = evaluate(model, test_loader)
print(f"Best val loss snapshot on test — loss={test_loss:.4f}  accuracy={test_acc:.4f}")

Epoch 01/15  train_loss=0.6419  train_acc_batch_mean=0.6261  val_loss=0.5746  val_acc=0.6839
Epoch 02/15  train_loss=0.5650  train_acc_batch_mean=0.7097  val_loss=0.5551  val_acc=0.7096
Epoch 03/15  train_loss=0.5314  train_acc_batch_mean=0.7320  val_loss=0.5368  val_acc=0.7146
Epoch 04/15  train_loss=0.4926  train_acc_batch_mean=0.7573  val_loss=0.4543  val_acc=0.7901
Epoch 05/15  train_loss=0.4587  train_acc_batch_mean=0.7851  val_loss=0.4371  val_acc=0.7858
Epoch 06/15  train_loss=0.4441  train_acc_batch_mean=0.7915  val_loss=0.4138  val_acc=0.8201
Epoch 07/15  train_loss=0.4161  train_acc_batch_mean=0.8089  val_loss=0.4226  val_acc=0.8082
Epoch 08/15  train_loss=0.3903  train_acc_batch_mean=0.8242  val_loss=0.3770  val_acc=0.8282
Epoch 09/15  train_loss=0.3817  train_acc_batch_mean=0.8313  val_loss=0.3795  val_acc=0.8189
Epoch 10/15  train_loss=0.3651  train_acc_batch_mean=0.8419  val_loss=0.3727  val_acc=0.8282
Epoch 11/15  train_loss=0.3526  train_acc_batch_mean=0.8461  val_loss=